# Online Purchase Prediction System

## Introduction & Problem Statement

- Online businesses often struggle to identify which website visitors are likely to complete a purchase.
- Most user sessions end without buying, which makes marketing and targeting decisions difficult and costly.
- Browsing behavior during a session (e.g., time spent, pages visited, bounce/exit rates, visitor type) contains signals of purchase intention.
- This project builds a supervised machine learning **binary classification** model to predict whether a user session will result in:
  - **Revenue = TRUE** (purchase completed), or
  - **Revenue = FALSE** (no purchase).
- The prediction is based on session-level browsing behavior and user/session characteristics.
- The model can support business decision-making such as:
  - identifying high-intent visitors,
  - improving conversion rate through personalized targeting,
  - optimizing marketing budget by focusing on likely buyers.

## Dataset Selection & Justification

- **Dataset Title:** Online Shoppers Intention Dataset (UCI Machine Learning Repository)
- **Source (Kaggle URL):**
  https://www.kaggle.com/datasets/henrysue/online-shoppers-intention
- **Dataset Type:** Structured tabular dataset (session-level records)
- **Size:** 12,330 user sessions
- **Features:** 18 attributes describing browsing behavior and session characteristics, including:
  - number of pages visited and time spent on different page categories,
  - bounce rate and exit rate,
  - visitor type and month,
  - other session-level attributes.
- **Target Variable:** `Revenue`
  - Binary label indicating whether the session resulted in a purchase (**TRUE**) or not (**FALSE**).

### Why this dataset?
- It directly matches the project goal: predicting purchase completion using session behavior.
- It is a real-world e-commerce dataset, making results meaningful and applicable.
- It contains a mix of **numerical and categorical** features, enabling comprehensive preprocessing and model comparison.
- It satisfies the project requirements:
  - Structured tabular dataset
  - Multiple features (≥ 10)
  - Large number of observations (thousands)
  - Binary classification target

### License Information
- Kaggle License: **Other (specified in description)** (as stated on the Kaggle dataset page).
- Original source: UCI Machine Learning Repository  
  https://archive.ics.uci.edu/ml/datasets/Online+Shoppers+Purchasing+Intention+Dataset

## Initial Data Inspection

In this section, we load the dataset and perform an initial inspection to understand:
- The dataset dimensions (rows and columns)
- Column names and data types
- Missing values (if any)
- Basic descriptive statistics for numerical features

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [14]:
df = pd.read_csv("online_shoppers_intention.csv")
df.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


In [15]:
df.shape

(12330, 18)

The dataset contains 12,330 observations (rows) and 18 features (columns).
Each row represents a unique user session, and each column represents a session-related attribute.

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Administrative           12330 non-null  int64  
 1   Administrative_Duration  12330 non-null  float64
 2   Informational            12330 non-null  int64  
 3   Informational_Duration   12330 non-null  float64
 4   ProductRelated           12330 non-null  int64  
 5   ProductRelated_Duration  12330 non-null  float64
 6   BounceRates              12330 non-null  float64
 7   ExitRates                12330 non-null  float64
 8   PageValues               12330 non-null  float64
 9   SpecialDay               12330 non-null  float64
 10  Month                    12330 non-null  object 
 11  OperatingSystems         12330 non-null  int64  
 12  Browser                  12330 non-null  int64  
 13  Region                   12330 non-null  int64  
 14  TrafficType           

The `df.info()` output provides a detailed structural summary of the dataset.

It confirms that the dataset contains 12,330 entries across 18 columns.
All columns have 12,330 non-null values, which indicates that there are no missing values in the dataset.

The dataset includes multiple data types:
- Integer features (int64)
- Floating-point features (float64)
- Categorical features (object)
- Boolean features (bool)

The presence of categorical and boolean variables (e.g., Month, VisitorType, Weekend, Revenue) suggests that encoding techniques will be required during the preprocessing stage before applying machine learning models.

Overall, the dataset is structurally complete and suitable for further exploratory data analysis.

In [17]:
df.isnull().sum()

,0
Administrative,0
Administrative_Duration,0
Informational,0
Informational_Duration,0
ProductRelated,0
ProductRelated_Duration,0
BounceRates,0
ExitRates,0
PageValues,0
SpecialDay,0


### Missing Values Analysis

To ensure data quality, a missing values check was performed using `df.isnull().sum()`.

The results indicate that all features contain zero missing values.

This confirms that the dataset is complete and does not require any imputation or missing-value handling before preprocessing.

In [18]:
df.describe()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,OperatingSystems,Browser,Region,TrafficType
count,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000
mean,2.315166,80.818611,0.503569,34.472398,31.731468,1194.746220,0.022191,0.043073,5.889258,0.061427,2.124006,2.357097,3.147364,4.069586
std,3.321784,176.779107,1.270156,140.749294,44.475503,1913.669288,0.048488,0.048597,18.568437,0.198917,0.911325,1.717277,2.401591,4.025169
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000
25%,0.000000,0.000000,0.000000,0.000000,7.000000,184.137500,0.000000,0.014286,0.000000,0.000000,2.000000,2.000000,1.000000,2.000000
50%,1.000000,7.500000,0.000000,0.000000,18.000000,598.936905,0.003112,0.025156,0.000000,0.000000,2.000000,2.000000,3.000000,2.000000
75%,4.000000,93.256250,0.000000,0.000000,38.000000,1464.157214,0.016813,0.050000,0.000000,0.000000,3.000000,2.000000,4.000000,4.000000
max,27.000000,3398.750000,24.000000,2549.375000,705.000000,63973.522230,0.200000,0.200000,361.763742,1.000000,8.000000,13.000000,9.000000,20.000000


The `df.describe()` output provides basic descriptive statistics for the numerical features in the dataset.

It summarizes:
- Mean
- Standard deviation
- Minimum and maximum values
- Quartiles (25%, 50%, 75%)

From this summary, we observe that:

- Duration-related features (e.g., Administrative_Duration, ProductRelated_Duration) show wide ranges, indicating variability in user browsing time.
- BounceRates and ExitRates are relatively small decimal values, representing proportions.
- Some features exhibit skewed distributions, which may require normalization during preprocessing.

This statistical overview helps in understanding the scale and distribution of numerical variables before moving to exploratory data analysis.

### Conclusion of Initial Inspection

The dataset has been successfully examined and verified.
It contains no missing values, includes a mixture of numerical and categorical features, and presents meaningful session-based attributes suitable for predictive modeling.

The dataset is now ready for detailed exploratory data analysis (EDA).